# Dataset Analysis

## Utils 

In [1]:
import igraph as ig 
from typing import Tuple
import numpy as np
import subprocess
import pandas as pd
import time as time

In [2]:
def check_number_in_file(file_path: str, number: int) -> Tuple[bool, str]:
    """
    Check if a given number is present in any line of the file.

    Parameters
    ----------
    file_path : str
        Path to the text file.
    number : int
        Number to check in the file.

    Returns
    -------
    bool
        True if the number is found in any line, False otherwise.
    """
    with open(file_path, 'r') as file:
        for line in file:
            if str(number) in line.split():
                return True, line
    return False, None

In [7]:
def import_graph(file_path: str) -> ig.Graph:
    """
    Import a graph from a txt file using igraph

    Parameters
    ----------
    file_path : str
        File path of the .txt file
        
    Returns
    -------
    ig.Graph
        Graph imported from the file path
    """
    if file_path.endswith(".txt"):
        graph = ig.Graph.Read_Edgelist(file_path, directed=False)
        graph = graph.simplify(multiple=True, loops=True)
    else:
        raise ValueError("File format not supported")
    
    # Igraph starts from 0 index, so if txt file starts from 1, we need to delete the first vertex
    if check_number_in_file(file_path, 0)[0] is False:
        graph.vs(0).delete()
    return graph

In [8]:
repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).strip().decode('utf-8')
dataset_root = f"{repo_root}/dataset/networks"

In [21]:
def graph_analysis(G: ig.Graph, graph_name: str) -> None:
    """
    Perform some analysis on the graph

    Parameters
    ----------
    G : ig.Graph
        Graph to analyze
    """
    networks_names = {
        "kar": "kar - Zachary Karate Club",
        "words": "words - David Copperfield Word Adjacency Network",
        "vote": "vote - Wikipedia Voting Network",
        "pow": "pow - U.S. Power Grid",
        "fb-75": "fb-75 - Facebook Friendship Network",
        "cond-mat": "cond-mat - Condense Matter Collaboration Network",

    }

    #Graph Analysis
    n_nodes = G.vcount()
    n_edges = G.ecount()
    degrees = G.degree()
    diameter = G.diameter()
    avg_degree = sum(degrees) / len(degrees)
    max_degree = max(degrees)
    avg_path_length = G.average_path_length()

    #Community Detection Analysis
    algorithms = {
        "Edge Betweenness": G.community_edge_betweenness,
        "Fast Greedy": G.community_fastgreedy,
        "Infomap": G.community_infomap,
        "Label Propagation": G.community_label_propagation,
        "Leading Eigenvector": G.community_leading_eigenvector,
        "Louvain": G.community_multilevel,
        "Spinglass": G.community_spinglass,
        "Walktrap": G.community_walktrap
    }
    results = []
    for name, algorithm in algorithms.items():
        #Avoid running some time-consuming algorithms on large graphs
        if n_nodes > 1000 and name == "Edge Betweenness":
            continue
        elif n_nodes > 10000 and name == "Spinglass":
            continue
        start_time = time.time()
        try:
            if name in ["Edge Betweenness", "Fast Greedy", "Walktrap"]:
                communities = algorithm().as_clustering()
            else:
                communities = algorithm()
            end_time = time.time()
            elapsed_time = round(end_time - start_time, 2)
            results.append({"Algorithm": name, "Number of Communities": len(communities), "Time (s)": elapsed_time})
        except ig.InternalError as e:
            print(f"Algorithm {name} failed: {e}")
            results.append({"Algorithm": name, "Number of Communities": "N/A", "Time (s)": "N/A"})
    df = pd.DataFrame(results)

    print("-"*20,"Graph Analysis","-"*20,"\n")
    print(f"Graph: {networks_names[graph_name]}")
    print(f"Number of nodes: {n_nodes}")
    print(f"Number of edges: {n_edges}")
    print(f"Average degree: {avg_degree:.2f}")
    print(f"Max degree: {max_degree}")
    print(f"Diameter: {diameter}")
    print(f"Average path length: {avg_path_length:.2f} \n")
    print("-"*13,"Community Detection Analysis","-"*13,"\n")
    print(df)

## Zachary Karate Club

In [9]:
graph_name = "kar"
kar = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(kar, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: kar - Zachary Karate Club
Number of nodes: 34
Number of edges: 78
Average degree: 4.59
Max degree: 17
Diameter: 5
Average path length: 2.41 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0     Edge Betweenness                      5  0.001886
1          Fast Greedy                      3  0.000222
2              Infomap                      3  0.005096
3    Label Propagation                      3  0.000062
4  Leading Eigenvector                      4  0.003677
5              Louvain                      4  0.000155
6            Spinglass                      4  0.109738
7             Walktrap                      5  0.000117


## David Copperfield Word Adjacency Network

In [95]:
graph_name = "words"
words = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(words, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: words - David Copperfield Word Adjacency Network
Number of nodes: 112
Number of edges: 425
Average degree: 7.59
Max degree: 49
Diameter: 5
Average path length: 2.54 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0     Edge Betweenness                     69  0.133481
1          Fast Greedy                      7  0.000235
2              Infomap                      2  0.005676
3    Label Propagation                      1  0.000070
4  Leading Eigenvector                     10  0.007696
5              Louvain                      8  0.000291
6            Spinglass                      8  0.598216
7             Walktrap                     25  0.000688


## Wikipedia Voting Network

In [96]:
graph_name = "vote"
vote = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(vote, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: vote - Wikipedia Voting Network
Number of nodes: 889
Number of edges: 2914
Average degree: 6.56
Max degree: 102
Diameter: 13
Average path length: 4.10 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities   Time (s)
0     Edge Betweenness                     16  29.051172
1          Fast Greedy                     14   0.005273
2              Infomap                     73   0.229745
3    Label Propagation                      7   0.001276
4  Leading Eigenvector                      6   0.030029
5              Louvain                      7   0.002762
6            Spinglass                     11   3.751297
7             Walktrap                     42   0.019040


## U.S. Power Grid

In [11]:
graph_name = "pow"
pow = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(pow, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: pow - U.S. Power Grid
Number of nodes: 4941
Number of edges: 6594
Average degree: 2.67
Max degree: 19
Diameter: 46
Average path length: 18.99 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities   Time (s)
0          Fast Greedy                     41   0.010860
1              Infomap                    488   1.471950
2    Label Propagation                    494   0.014400
3  Leading Eigenvector                    126   3.195802
4              Louvain                     41   0.006394
5            Spinglass                     25  17.672550
6             Walktrap                    364   0.084916


## Facebook Friendship Network

In [18]:
graph_name = "fb-75"
fb = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(fb, graph_name)

Algorithm Spinglass failed: Error at src/community/spinglass/clustertool.cpp:293: Cannot work with unconnected graph. -- Invalid value
-------------------- Graph Analysis -------------------- 

Graph: fb-75 - Facebook Friendship Network
Number of nodes: 6386
Number of edges: 217662
Average degree: 68.17
Max degree: 930
Diameter: 9
Average path length: 2.77 

------------- Community Detection Analysis ------------- 

             Algorithm Number of Communities Time (s)
0          Fast Greedy                    24     1.16
1              Infomap                   140    10.15
2    Label Propagation                    17     0.04
3  Leading Eigenvector                    13     0.67
4              Louvain                    18     0.21
5            Spinglass                   N/A      N/A
6             Walktrap                   357     6.19


## Condense Matter Collaboration Network

In [22]:
graph_name = "cond-mat"
cond_mat = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(cond_mat, graph_name)

/Users/silver22/anaconda3/lib/python3.11/site-packages/igraph/community.py:98: RuntimeWarning: ARPACK solver failed to converge (10001 iterations, 0/1 eigenvectors converged) at src/linalg/arpack.c:899
  membership, _, q = GraphBase.community_leading_eigenvector(graph, clusters, **kwds)


Algorithm Leading Eigenvector failed: Error at src/community/leading_eigenvector.c:563: ARPACK did not converge. -- No eigenvalues to sufficient accuracy
-------------------- Graph Analysis -------------------- 

Graph: cond-mat - Condense Matter Collaboration Network
Number of nodes: 108299
Number of edges: 93439
Average degree: 1.73
Max degree: 279
Diameter: 15
Average path length: 5.35 

------------- Community Detection Analysis ------------- 

             Algorithm Number of Communities Time (s)
0          Fast Greedy                 86002     1.93
1              Infomap                 86975    45.04
2    Label Propagation                 87261     0.33
3  Leading Eigenvector                   N/A      N/A
4              Louvain                 85786      0.2
5             Walktrap                 88072    15.74
